# Work Package 1 — Single Tissue Channel × Exp-4 (Late Fusion, Concat) — Colab Run

Advisor request: repeat Experiment 4 (late fusion, concat) with exactly **one** tissue
probability channel at a time — granulation, slough, necrosis, tendon, bone — instead
of all five fused together, to check whether any individual channel's signal is
obscured when all five are fused.

Everything except the tissue branch's input is **unchanged** from exp-4's own
`concat` run: same encoder (MiT-B1, ImageNet init), same CORN head, same
single-phase protocol (`freeze_epochs=0`), same lr/epochs/augmentation, same
`datasets/wound-staging-masks-exp-3/` 8-channel `.npy` data and the same verified
PIID 3-fold split exp-3/4/5 already use. Only the tissue CNN's input channel count
changes: 1 selected channel instead of 5 concatenated.

Writes **one shared** `results.csv` across all 5 channels × 3 folds
(`config_id = mit_b1__corn__wp1_exp4concat__tissue={channel}`), skip-if-already-done,
safe to interrupt and resume — matches exp-4's own `train.py` resume pattern.

**Reference numbers to compare against** (already recorded, not re-derived here):
- RGB baseline (task_3, 3-fold mean): accuracy 0.8168, macro-F1 0.8182, QWK 0.9138
- Exp-4 concat, all 5 tissue channels (3-fold mean): accuracy 0.7983, macro-F1 0.8003, QWK 0.9109

If a single channel beats 0.7983/0.8003/0.9109, that channel carries signal the
5-channel fusion is diluting. If every single channel is close to or below the
5-channel number, the full fusion isn't obscuring anything — there's just not much
extra information in any one channel alone.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Clone / update the repo

In [ ]:
import os

REPO_DIR = "/content/CanRollasThesisProject"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/canrollas/CanRollasThesisProject.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

## 3. Fetch the dataset

Same `wound-staging-masks-exp-3.zip` as exp-3/exp-4/exp-5 — nothing new to upload
if you already put it on Drive for those runs.

In [ ]:
import glob
import os

ZIP_PATH = "/content/drive/MyDrive/thesis/wound-staging-masks-exp-3.zip"
EXTRACT_DIR = "/content/data"
DATASET_NAME = "wound-staging-masks-exp-3"
DATASET_DIR = f"{REPO_DIR}/datasets/{DATASET_NAME}"

if not glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    !unzip -q "{ZIP_PATH}" -d {EXTRACT_DIR}

candidates = glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True)
assert candidates, f"couldn't find an extracted '{DATASET_NAME}' folder under {EXTRACT_DIR}"
source = candidates[0]
print("dataset found at:", source)

os.makedirs(os.path.dirname(DATASET_DIR), exist_ok=True)
if os.path.islink(DATASET_DIR):
    os.remove(DATASET_DIR)
os.symlink(source, DATASET_DIR)
!ls -la {DATASET_DIR}

## 4. Install dependencies

In [ ]:
import os
!pip install -q -r {REPO_DIR}/task_3/requirements.txt
os.chdir(f"{REPO_DIR}/more_experiments/experiment_extended/wp1_single_tissue_channel/exp4_concat/scripts")
print(os.getcwd())

## 4b. Smoke-test sanity check

This WP reuses exp-4's own already shape/gradient-verified `model.py` essentially
unchanged — only the tissue arm's input channel count differs (1 instead of 5) — so
no separate `test_*.py` was written for it (kept to the brief's "minimal
modification" instruction). Instead, run `train.py`'s built-in `--smoke` mode (1
fold, 2 epochs, 8/4/4 samples) once to catch any shape/IO issue before committing to
the full 15-run sweep.

In [ ]:
!python train.py --tissue-channel granulation --smoke --workers 2

## 5. Redirect checkpoints/results to Drive

In [ ]:
import yaml

OUTPUT_DIR = "/content/drive/MyDrive/thesis/wp1_exp4concat_outputs"

with open("../configs/base.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["checkpoint_dir"] = f"{OUTPUT_DIR}/checkpoints"
cfg["results_csv"] = f"{OUTPUT_DIR}/results/results.csv"
with open("../configs/base.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("checkpoint_dir:", cfg["checkpoint_dir"])
print("results_csv:   ", cfg["results_csv"])

## 6. Train — sweep all 5 tissue channels × 3 folds

`--tissue-channel all` (the default) sweeps granulation → slough → necrosis →
tendon → bone, 3 folds each (15 runs total). `train.py` already skips any
`(config_id, fold)` pair already present in `results.csv`, so the retry loop below
is safe exactly like exp-4's own notebook: if Colab disconnects mid-sweep, just
re-run this cell — checkpoints/results already point at Drive from step 5, so
nothing already finished is repeated.

In [ ]:
import subprocess
import time

cmd = ["python", "-u", "train.py", "--tissue-channel", "all", "--workers", "4"]
while True:
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    returncode = proc.wait()
    if returncode == 0:
        print("single-tissue-channel exp-4 sweep finished")
        break
    print(f"train.py exited with code {returncode}, retrying in 10s...")
    time.sleep(10)

## 6b. (Optional) run a single channel only

Useful if you want to split the sweep across multiple Colab sessions/GPUs instead
of one long run. Set `CHANNEL` and run this cell instead of 6 above.

In [ ]:
CHANNEL = "bone"  # one of: granulation, slough, necrosis, tendon, bone

cmd = ["python", "-u", "train.py", "--tissue-channel", CHANNEL, "--workers", "4"]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
print("exit code:", proc.wait())

## 7. Check results

In [ ]:
import pandas as pd

df = pd.read_csv(f"{OUTPUT_DIR}/results/results.csv")
df["tissue_channel"] = df["config_id"].str.extract(r"tissue=(\w+)")
display(df)

agg = df.groupby("tissue_channel")[["accuracy", "macro_f1", "mae", "qwk"]].agg(["mean", "std"])
display(agg)

print("\nReference (already recorded, not re-derived here):")
print("  RGB baseline (3-fold mean):        accuracy=0.8168 macro_f1=0.8182 qwk=0.9138")
print("  Exp-4 concat, all 5 channels:       accuracy=0.7983 macro_f1=0.8003 qwk=0.9109")